# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Run the ASV pipeline (Modules 1 & 2)

In [ ]:

./asv_pipeline.sh --config asv.conf

# Basic Stats
THREADS="$(nproc)"
seqkit stat -a -T -o ../spark_refactored_output/stats/fastq_stats.tsv -j ${THREADS} ../fastq_combined/*.fastq.gz
seqkit stat -a -T -o ../spark_refactored_output/stats/fastp_fastqs.tsv -j ${THREADS} ../spark_refactored_output/fastp/*.fastq.gz
seqkit stat -a -T -o ../spark_refactored_output/stats/filtered_fastqs.tsv -j ${THREADS} ../spark_refactored_output/filtered/*.fasta
seqkit stat -a -T -o ../spark_refactored_output/stats/concat_fastas.tsv -j ${THREADS} ../spark_refactored_output/concat/concat.fasta

### Run QIIME2 Taxonomic Classifier (Module 3)

In [ ]:
conda activate qiime2-amplicon-2024.10
mkdir -p ../spark_refactored_output/taxonomy
awk '/^>/ {print; next} {print toupper($0)}' ../spark_refactored_output/ASVs/ASVs.fasta > ../spark_refactored_output/ASVs/ASVs.upper.fasta
python qiime_vs_classifier.py \
  --input-fasta ../spark_refactored_output/ASVs/ASVs.upper.fasta \
  --ref-taxonomy ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output ../spark_refactored_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Decontamination/Non-target Identification (Module 4)

In [ ]:
mamba activate asv-py

rm -rf ../spark_refactored_output/mito
mkdir -p ../spark_refactored_output/mito/mitomap
rm -rf ../spark_refactored_output/ASVs/chunks

# Chunk up the FASTA so MITOMASTER can be sped up
seqkit split -s 10 -O ../spark_refactored_output/ASVs/chunks ../spark_refactored_output/ASVs/ASVs_filtered.fasta

python ./mitomaster.py \
       --data-dir ../spark_refactored_output/ASVs/chunks/ \
       --output-file ../spark_refactored_output/mito/mitomap/mitomaster_output.tsv

blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASVs_filtered.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/mito_ncbi \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/mito/mitomap/mito_ncbi.blast6.tsv

blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASVs_filtered.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/ssu_pipeline_contaminants \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv

python ./mito_checker.py \
       --mitomaster-file ../spark_refactored_output/mito/mitomap/mitomaster_output.tsv \
       --mito-blast ../spark_refactored_output/mito/mitomap/mito_ncbi.blast6.tsv \
       --silva-tax ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
       --biof-file ../spark_refactored_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv \
       --output-dir ../spark_refactored_output/mito/mitomap/ --overwrite

### ASV Final Cleaning (Module 5)

In [ ]:
mkdir -p ../spark_refactored_output/mito/ASVs
python filter_nontarget.py \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASV_filtered.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/mito/mitomap/nontarget.master.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/ASVs/ASV_target.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    0.005

## 2. Downstream analysis pipeline (Module 6)
This section reviews the steps involved in creating the tables and figures from the final ASV data set and associated metadata.

*NOTE: must run ASV pipeline first, as all data products are required.*

### Build Data Loss Sankey Diagram

In [ ]:
python sankey_builder.py \
    --data-dir ../ \
    --sub-dir spark_combined_output_reduced \
    --metadata ../spark_combined_output_reduced/Sample_metadata.tsv \
    --fastq-stats M1_quality_control/stats/fastq_stats.tsv \
    --filtered-stats M1_quality_control/stats/filtered_fastqs.tsv \
    --asv-raw M2_error_correction/ASVs/ASV_counts.tsv \
    --asv-decon M5_cleaned_counts/ASVs/ASV_target.decon.tsv \
    --asv-micro M5_cleaned_counts/ASVs/ASV_target.micro.tsv \
    --samp-col sample \
    --type-col type_group \
    --make-labeled --make-unlabeled

### Plot Metadata

In [ ]:
python plot_metadata.py \
    --data-dir ../ \
    --sub-dir spark_combined_output_reduced \
    --metadata ../spark_combined_output_reduced/Sample_metadata.tsv \
    --fastq-stats ../spark_combined_output_reduced/M1_quality_control/stats/fastq_stats.tsv \
    --taxonomy ../spark_combined_output_reduced/M3_taxonomic_annotation/ASV_SILVA_tax.full-length.vsearch.tsv \
    --asv-micro ../spark_combined_output_reduced/M5_cleaned_counts/ASVs/ASV_target.micro.tsv \
    --asv-mito ../spark_combined_output_reduced/M5_cleaned_counts/ASVs/ASV_target.mito.tsv \
    --make-micro --make-mito

python outlier_checker.py \
    --data-dir ../ \
    --metadata spark_combined_output_reduced/metadata/metadata_updated_micro.tsv \
    --output spark_combined_output_reduced/metadata \
    --asv spark_combined_output_reduced/ASVs/ASV_target.micro.tsv \
    --group-cols type_group

### Plot Upset

In [ ]:

python plot_upset.py \
    --data-dir ../ \
    --subdir spark_refactored_output \
    --domain both \
    --taxonomy-path ../spark_refactored_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
    --formats svg,pdf --do-composite-oral-lung

# Cancer Status Venns
python plot_upset_general.py \
    --counts-final ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/ASV_final.micro.tsv \
    --metadata ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/metadata_updated.tsv \
    --sample-col orig_sample \
    --group-col status \
    --out-dir ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/ \
    --formats svg,pdf \
    --suite '{"name": "cancer_status","counts": "final","groups": ["Cancer", "Non-Cancer"],"plots": ["upset", "upset_weighted", "venn"],"tables": true}' \
    --palette '{"Non-Cancer": "#F2F2F2","Cancer":"#FF0000"}'

python venn_bubbles.py \
    --data-dir ../ \
    --subdir spark_refactored_output \
    --asv-meta ../spark_refactored_output/metadata/ASV_meta_micro.tsv \
    --presence ../spark_refactored_output/metadata/Three_types_micro_Three_types_venn_presence_table.tsv \
    --type-order "Oral Rinse,BAL,Lung Brush" --formats svg,pdfs

### Run Alpha and Beta Diversity

In [ ]:

python collectors_curve.py \
    --counts ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/ASV_final.micro.tsv \
    --meta ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/metadata_updated.tsv \
    --sample-id-col lmp_id \
    --group-col type_group \
    --out_prefix ../spark_combined_output_reduced/M6_downstream_analysis/diversity/ \
    --permutations 999 --seed 42 \
    --group-colors "Oral Rinse=#6A3D9A,BAL=#0072B2,Lung Brush=#009E73" \
    --group-order "Oral Rinse,BAL,Lung Brush" \
    --formats pdf,svg

python calc_div.py \
    --micro-table ../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --mito-table ../spark_refactored_output/mito/ASVs/ASV_final.mito.tsv \
    --outdir ../spark_refactored_output/diversity \
    --mito-outdir ../spark_refactored_output/mito/diversity 

python plot_diversity.py \
    --metadata ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/metadata_updated.tsv \
    --master ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/master_table.tsv \
    --alpha ../spark_combined_output_reduced/M6_downstream_analysis/diversity/tables/shannon.tsv \
    --bray ../spark_combined_output_reduced/M6_downstream_analysis/diversity/tables/bray.tsv \
    --jacc ../spark_combined_output_reduced/M6_downstream_analysis/diversity/tables/jaccard.tsv \
    --outliers-all ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/outliers_table.tsv \
    --type-order "Oral Rinse,BAL,Lung Brush" \
    --mito-alpha ../spark_combined_output_reduced/M6_downstream_analysis/diversity/mitochondrial/diversity/tables/shannon.mito.tsv \
    --mito-bray ../spark_combined_output_reduced/M6_downstream_analysis/diversity/mitochondrial/diversity/tables/bray.mito.tsv \
    --mito-jacc ../spark_combined_output_reduced/M6_downstream_analysis/diversity/mitochondrial/diversity/tables/jaccard.mito.tsv \
    --outdir ../spark_combined_output_reduced/M6_downstream_analysis/diversity/ \
    --mito-outdir ../spark_combined_output_reduced/M6_downstream_analysis/diversity/mitochondrial/diversity/ \
    --exclude-types "Skin Brush,Scope Flush"

### Plot Clustermaps

In [ ]:
python plot_clustermaps.py \
    --asv-meta ../spark_refactored_output/metadata/ASV_meta_micro.tsv \
    --metadata ../spark_refactored_output/metadata/metadata_updated_micro.tsv \
    --isa ../spark_refactored_output/indicspecies/Type_status_ISA_results.tsv \
    --outdir ../spark_refactored_output/diversity \
    --type-order "Oral Rinse,BAL,Lung Brush" \
    --exclude-types "Skin Brush,Scope Flush" \
    --mito-asv ../spark_refactored_output/mito/ASVs/ASV_final.mito.tsv \
    --mito-outdir ../spark_refactored_output/mito/diversity \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3"
    

### Run indicspecies (R)

In [ ]:
Rscript run_indicspecies.R \
    --asv /home/ryan/Projects/UBC/LMP/spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/ASV_final.micro.tsv \
    --meta /home/ryan/Projects/UBC/LMP/spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/metadata_updated.tsv \
    --sample-col sample \
    --group-cols status,type_group \
    --outdir ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies2 \
    --participant-col Participant_ID

### Plot indicspecies Results

In [ ]:
# Run DULEG output
python plot_indicspecies.py \
    --type-results ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/tables/type_group_indicator_species_results_DULEG.tsv \
    --status-results ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/tables/status_indicator_species_results_DULEG.tsv \
    --type-venn ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/Three_types_venn_presence_table.tsv \
    --status-venn ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/cancer_status_cancer_status_venn_presence_table.tsv \
    --taxonomy ../spark_combined_output_reduced/M3_taxonomic_annotation/ASV_SILVA_tax.full-length.vsearch.tsv \
    --outdir ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/ \
    --type-index  "1=BAL,2=Lung Brush,3=Oral Rinse,4=BAL+Lung Brush,5=BAL+Oral Rinse,6=Lung Brush+Oral Rinse,7=Oral Rinse+BAL+Lung Brush" \
    --status-index "1=Cancer,2=Non-Cancer,3=Cancer+Non-Cancer" \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3" \
    --status-markers "Non-Cancer=D,Cancer=X,Cancer+Non-Cancer=^,not_indicator=o" \
    --label-col "ASV_ID"

    
# OR Run with Multi-level ISA (Recomended)
python plot_indicspecies.py \
    --type-results ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/tables/type_group_indicator_species_results.tsv \
    --status-results ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/tables/status_indicator_species_results.tsv \
    --type-venn ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/Three_types_venn_presence_table.tsv \
    --status-venn ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/cancer_status_cancer_status_venn_presence_table.tsv \
    --taxonomy ../spark_combined_output_reduced/M3_taxonomic_annotation/ASV_SILVA_tax.full-length.vsearch.tsv \
    --outdir ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/ \
    --type-index  "1=BAL,2=Lung Brush,3=Oral Rinse,4=BAL+Lung Brush,5=BAL+Oral Rinse,6=Lung Brush+Oral Rinse,7=Oral Rinse+BAL+Lung Brush" \
    --status-index "1=Cancer,2=Non-Cancer,3=Cancer+Non-Cancer" \
    --type-palette "Oral Rinse=#6A3D9A,BAL+Oral Rinse=#F19CBB,BAL=#0072B2,BAL+Lung Brush=#00FFFF,Lung Brush=#009E73,Lung Brush+Oral Rinse=#C1EAAD,Oral Rinse+BAL+Lung Brush=#000000,not_indicator=#D3D3D3" \
    --status-palette "Non-Cancer=#FFFFFF,Cancer=#A50026,Cancer+Non-Cancer=#000000,not_indicator=#D3D3D3" \
    --status-markers "Non-Cancer=D,Cancer=X,Cancer+Non-Cancer=^,not_indicator=o" \
    --label-col "ASV_ID" \
    --p-thresh 1000000000000

### Run SPIEC-EASI (R)

In [ ]:
Rscript run_spieceasi.R \
    --counts=../spark_refactored_output/ASVs/ASV_final.micro.tsv \
    --outdir=../spark_refactored_output/spieceasi \
    --force-graphs TRUE

### Graph Network

In [ ]:
python graph_network.py \
    --data-dir ../spark_combined_output_reduced/M6_downstream_analysis/ \
    --outdir ../spark_combined_output_reduced/M6_downstream_analysis/spieceasi \
    --graph-pos-all ../spark_combined_output_reduced/M6_downstream_analysis/spieceasi/tables/network_pos_all.graphml \
    --graph-pos-sub ../spark_combined_output_reduced/M6_downstream_analysis/spieceasi/tables/network_pos_sub.graphml \
    --node-features ../spark_combined_output_reduced/M6_downstream_analysis/spieceasi/tables/node_features.csv \
    --asv-counts ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/ASV_final.micro.tsv \
    --taxonomy ../spark_combined_output_reduced/M3_taxonomic_annotation/ASV_SILVA_tax.full-length.vsearch.tsv \
    --venn ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/Three_types_venn_presence_table.tsv \
    --type-summary ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/tables/type_group_indicator_species_summary.tsv \
    --status-summary ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies/tables/status_indicator_species_summary.tsv
    

### Build ASV Downstream Master Summary

In [ ]:
python summary/build_master_asv_summary.py \
    --asv-meta ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/ASV_meta.tsv \
    --asv-counts ../spark_combined_output_reduced/M6_downstream_analysis/metadata_summaries/tables/ASV_final.tsv \
    --clustermaps-dir ../spark_combined_output_reduced/M6_downstream_analysis/clustermaps \
    --indicspecies-dir ../spark_combined_output_reduced/M6_downstream_analysis/indicspecies \
    --spieceasi-dir ../spark_combined_output_reduced/M6_downstream_analysis/spieceasi \
    --whitelist "Type_status_ISA_results.tsv,Type_status_Venn_results.tsv,node_features.status.tsv,node_features.type.tsv" \
    --outdir ../spark_combined_output_reduced/M6_downstream_analysis/summary/tables2